In [ ]:
# ──────────────────────────────────────────────────
# [셀 1] 필수 패키지 설치
# selenium        : 브라우저 자동화 (크롤링)
# webdriver-manager: ChromeDriver 자동 설치 및 관리
# pandas           : 데이터 정리 및 CSV 저장
# ──────────────────────────────────────────────────
import sys
!{sys.executable} -m pip install selenium webdriver-manager pandas -q

In [ ]:
# ──────────────────────────────────────────────────
# [셀 2] 라이브러리 import 및 전역 설정값 정의
# ──────────────────────────────────────────────────
import time
import random
import json
import pandas as pd
from datetime import datetime

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ── 수집 범위 설정 ────────────────────────────────
MAX_PRODUCTS        = 50    # 이번 실행에서 수집할 최대 제품 수
START_PRODUCT       = 650   # 건너뛸 제품 수 (이어서 크롤링할 때 활용)
REVIEWS_PER_PRODUCT = 200   # 제품당 최대 수집 리뷰 수

# ── 요청 간격 설정 (봇 탐지 방지용 랜덤 딜레이) ──
DELAY_MIN = 1.5
DELAY_MAX = 3.0

# ── 출력 파일 경로 ────────────────────────────────
OUTPUT_CSV  = "oliveyoung_cream_reviews_650-700.csv"
OUTPUT_JSON = "oliveyoung_cream_reviews_650-700.json"

print("✅ 설정 완료")


In [ ]:
# ──────────────────────────────────────────────────
# [셀 3] Chrome WebDriver 관련 유틸 함수 정의
# ──────────────────────────────────────────────────

def create_driver():
    """봇 탐지를 우회하도록 설정된 Chrome WebDriver를 생성하여 반환한다."""
    options = Options()
    # 자동화 탐지 플래그 비활성화 (사이트의 봇 차단 우회)
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    # 리눅스/서버 환경에서 크래시 방지 옵션
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1280,900")
    # 실제 브라우저처럼 보이도록 User-Agent 설정
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    # webdriver 속성을 undefined로 덮어써서 탐지 방지
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    return driver

def sleep_random():
    """DELAY_MIN ~ DELAY_MAX 사이 랜덤 시간만큼 대기한다. (요청 간격 조절)"""
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

def close_popup(driver):
    """화면에 나타난 팝업/모달 닫기 버튼을 자동으로 클릭한다."""
    try:
        btns = driver.find_elements(By.CSS_SELECTOR,
            ".layer-wrap .btn-close, .popup-close, .modal-close"
        )
        for btn in btns:
            if btn.is_displayed():
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(0.5)
    except:
        pass

print("✅ 드라이버 함수 정의 완료")


In [ ]:
# ──────────────────────────────────────────────────
# [셀 4] 제품 목록 수집 함수
# 올리브영 크림 카테고리 페이지를 페이지네이션으로 순회하며
# 제품명, 브랜드, URL, 상품번호, 이미지 URL을 수집한다.
# ──────────────────────────────────────────────────

def get_product_list(driver, max_products=MAX_PRODUCTS, start_product=START_PRODUCT):
    """
    올리브영 크림 카테고리 목록을 페이지 단위로 순회하며 제품 정보를 수집한다.

    Args:
        driver        : Selenium WebDriver 인스턴스
        max_products  : 수집할 최대 제품 수
        start_product : 건너뛸 제품 수 (이어서 크롤링 시 사용)

    Returns:
        list[dict]: 제품 정보 딕셔너리 리스트
    """
    products = []
    page = 1
    skipped = 0  # 건너뛴 제품 수 카운터

    print("=" * 60)
    print("▶ 1단계: 제품 목록 수집 시작")
    print(f"  건너뛸 제품: {start_product}개 / 수집할 제품: {max_products}개")
    print("=" * 60)

    while True:
        # 카테고리 목록 URL (크림 카테고리, 인기순 정렬, 24개씩)
        url = (
            "https://www.oliveyoung.co.kr/store/display/getMCategoryList.do"
            f"?dispCatNo=100000100010015&fltDispCatNo=&prdSort=01"
            f"&pageIdx={page}&rowsPerPage=24"
        )
        print(f"  [페이지 {page}] 요청 중...")
        driver.get(url)
        time.sleep(3)

        # 스크롤을 내렸다 올려서 lazy-load 이미지까지 로딩 유도
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(1)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        close_popup(driver)

        # 제품 카드가 1개 이상 로딩될 때까지 최대 15초 대기
        try:
            WebDriverWait(driver, 15).until(
                lambda d: len(d.find_elements(By.CSS_SELECTOR, "#Contents .prd_info")) >= 1
            )
        except:
            print("  [완료] 마지막 페이지")
            break

        cards = driver.find_elements(By.CSS_SELECTOR, "#Contents .prd_info")
        if not cards:
            break

        for card in cards:
            try:
                link = card.find_element(By.CSS_SELECTOR, "a.prd_thumb")
                product_url = link.get_attribute("href") or ""
                if not product_url:
                    continue

                # START_PRODUCT 이전 제품은 건너뜀 (이어서 수집 시 사용)
                if skipped < start_product:
                    skipped += 1
                    continue

                # URL 쿼리스트링에서 goodsNo 파라미터 추출
                goods_no = ""
                if "goodsNo=" in product_url:
                    goods_no = product_url.split("goodsNo=")[1].split("&")[0]

                # 브랜드명 추출 (없으면 빈 문자열)
                try:
                    brand = card.find_element(By.CSS_SELECTOR, "span.tx_brand").text.strip()
                except:
                    brand = ""

                # 제품명 추출 (없으면 빈 문자열)
                try:
                    product_name = card.find_element(By.CSS_SELECTOR, "p.tx_name").text.strip()
                except:
                    product_name = ""

                # 제품 썸네일 이미지 URL 추출 (src 우선, 없으면 data-src)
                try:
                    img = link.find_element(By.CSS_SELECTOR, "img")
                    image_url = img.get_attribute("src") or img.get_attribute("data-src") or ""
                except:
                    image_url = ""

                products.append({
                    "product_name": product_name,
                    "brand":        brand,
                    "product_url":  product_url,
                    "goods_no":     goods_no,
                    "image_url":    image_url,
                })

                # 최대 수집 수에 도달하면 즉시 반환
                if max_products and len(products) >= max_products:
                    print(f"  [완료] 최대 제품 수({max_products}개) 도달")
                    return products

            except Exception as e:
                print(f"  [WARN] 파싱 오류: {e}")
                continue

        print(f"  → 누적 수집: {len(products)}개 (건너뜀: {skipped}개)")
        sleep_random()
        page += 1

    print(f"\n총 {len(products)}개 제품 수집 완료\n")
    return products

print("✅ 제품 목록 함수 정의 완료")

In [ ]:
# ──────────────────────────────────────────────────
# [셀 5] 리뷰 수집 함수 (Shadow DOM + 스크롤 방식)
# 올리브영 리뷰는 Shadow DOM 기반 Web Component로 렌더링되므로
# JavaScript를 직접 실행해 shadowRoot 내부 데이터를 추출한다.
# scroll_and_collect: 스크롤하며 뷰포트에 렌더링된 리뷰를 점진 수집
# get_reviews_for_product: 제품 페이지로 이동 후 리뷰 탭 진입
# ──────────────────────────────────────────────────


In [ ]:
# ──────────────────────────────────────────────────
# [셀 6] 메인 실행 블록 - 제품 목록 수집 → 리뷰 수집
# ──────────────────────────────────────────────────

print(f"\n{'='*60}")
print("  OVERFIT 프로젝트 - 올리브영 스킨/토너 리뷰 크롤러")
print(f"  실행 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*60}\n")

driver = create_driver()
all_reviews = []

try:
    # 1단계: 제품 목록 수집
    products = get_product_list(driver, max_products=MAX_PRODUCTS, start_product=START_PRODUCT)

    if not products:
        print("[ERROR] 제품 목록 수집 실패")
    else:
        print("=" * 60)
        print("▶ 2단계: 제품별 리뷰 수집 시작")
        print("=" * 60)

        # 각 제품에 대해 순서대로 리뷰 수집
        for i, product in enumerate(products, 1):
            print(f"\n  [{i}/{len(products)}] {product['brand']} - {product['product_name']}")
            reviews = get_reviews_for_product(driver, product, max_reviews=REVIEWS_PER_PRODUCT)
            print(f"    → {len(reviews)}개 리뷰 수집")
            all_reviews.extend(reviews)
            sleep_random()  # 제품 간 랜덤 딜레이

        print(f"\n총 {len(all_reviews)}개 리뷰 수집 완료")

finally:
    # 예외 발생 여부와 관계없이 브라우저는 반드시 종료
    driver.quit()
    print("\n브라우저 종료")

In [ ]:
# ──────────────────────────────────────────────────
# [셀 7] 수집 결과 저장 및 기본 통계 출력
# CSV와 JSON 두 가지 형식으로 저장한다.
# ──────────────────────────────────────────────────

# 컬럼 순서를 명시하여 DataFrame 생성
df = pd.DataFrame(all_reviews, columns=[
    "product_name", "brand", "product_url", "product_image_url",
    "rating", "review_text", "review_url", "review_image_url",
])

# CSV 저장 (utf-8-sig: 엑셀에서 한글 깨짐 방지)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"✅ CSV 저장 완료: {OUTPUT_CSV} ({len(df)}행)")

# JSON 저장 (ensure_ascii=False: 한글 유니코드 이스케이프 방지)
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(all_reviews, f, ensure_ascii=False, indent=2)
print(f"✅ JSON 저장 완료: {OUTPUT_JSON}")

# 수집 결과 기본 통계
print(f"\n─── 통계 ───")
print(f"  총 리뷰 수       : {len(df)}")
print(f"  총 제품 수       : {df['product_name'].nunique()}")
print(f"  평균 별점        : {df['rating'].mean():.2f}")
print(f"  이미지 있는 리뷰 : {(df['review_image_url'] != '').sum()}")

# 상위 10개 리뷰 미리보기
df.head(10)